# 12_Gold_Condition

Gold-layer notebook. Attach `FHIR_Lakehouse` before running.

In [1]:
from pyspark.sql.functions import col, element_at, regexp_extract, to_date, current_timestamp
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

StatementMeta(, c3f79e1b-82c2-4dbc-be42-d41e27e2c8f7, 3, Finished, Available, Finished, False)

DataFrame[]

In [2]:
silver_condition_df = spark.table("silver.condition")
print("Silver Condition Records:", silver_condition_df.count())

StatementMeta(, c3f79e1b-82c2-4dbc-be42-d41e27e2c8f7, 4, Finished, Available, Finished, False)

Silver Condition Records: 11878


In [3]:
gold_condition_df = silver_condition_df.select(
    col("id").alias("condition_id"),
    col("subject.reference").alias("patient_reference"),
    regexp_extract(col("subject.reference"), r"Patient/([^/]+)", 1).alias("patient_id"),
    col("encounter.reference").alias("encounter_reference"),
    col("clinical_status").alias("clinical_status"),
    col("clinical_status_display").alias("clinical_status_display"),
    element_at(col("verificationStatus.coding"), 1).getField("display").alias("verification_status"),
    col("code.text").alias("condition_name"),
    element_at(col("code.coding"), 1).getField("code").alias("condition_code"),
    element_at(col("code.coding"), 1).getField("display").alias("condition_display"),
    to_date(col("onsetDateTime")).alias("onset_date"),
    to_date(col("recordedDate")).alias("recorded_date"),
    to_date(col("abatementDateTime")).alias("abatement_date"),
    col("meta.lastUpdated").alias("source_last_updated"),
    col("extraction_timestamp"),
    col("load_timestamp")
).withColumn("gold_load_timestamp", current_timestamp())

StatementMeta(, c3f79e1b-82c2-4dbc-be42-d41e27e2c8f7, 5, Finished, Available, Finished, False)

In [4]:
(gold_condition_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable("gold.fact_condition"))

StatementMeta(, c3f79e1b-82c2-4dbc-be42-d41e27e2c8f7, 6, Finished, Available, Finished, False)

In [5]:
spark.table("gold.fact_condition").printSchema()
spark.table("gold.fact_condition").show(10, truncate=False)
print("Total Gold Condition Records:", spark.table("gold.fact_condition").count())

StatementMeta(, c3f79e1b-82c2-4dbc-be42-d41e27e2c8f7, 7, Finished, Available, Finished, False)

root
 |-- condition_id: string (nullable = true)
 |-- patient_reference: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- encounter_reference: string (nullable = true)
 |-- clinical_status: string (nullable = true)
 |-- clinical_status_display: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- condition_name: string (nullable = true)
 |-- condition_code: string (nullable = true)
 |-- condition_display: string (nullable = true)
 |-- onset_date: date (nullable = true)
 |-- recorded_date: date (nullable = true)
 |-- abatement_date: date (nullable = true)
 |-- source_last_updated: string (nullable = true)
 |-- extraction_timestamp: timestamp (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- gold_load_timestamp: timestamp (nullable = true)

+------------------------------------+--------------------------------------------+------------------------------------+-------------------+---------------+----------------------